# Importaciones

In [1]:
include("dependencies.jl")
include("helpers.jl")
include("wrappers.jl")
# Cargamos los datos preparados en el notebook anterior al instante
JLD2.@load "datos_procesados.jld2" df_trainval df_test X_trainval y_trainval folds_trainval n_features

6-element Vector{Symbol}:
 :df_trainval
 :df_test
 :X_trainval
 :y_trainval
 :folds_trainval
 :n_features

# Modelos básicos y selección de atributos (20%)

In [2]:
# Definición de diccionarios de configuración con MLP corregido
dic_filtros = Dict(
    "ANOVA" => MyANOVAFilter(n_features=n_features),
    "Pearson" => MyPearsonFilter(n_features=n_features),
    "Spearman" => MySpearmanFilter(n_features=n_features),
    "Kendall" => MyKendallFilter(n_features=n_features),
    "MI" => MyMIFilter(n_features=n_features),
    "RFE" => MyRFEFilter(n_features=n_features)
)

dic_reducciones = Dict(
    "Sin reducción" => nothing,
    "PCA" => PCA(variance_ratio=0.95),
    "ICA" => ICA(),
    "LDA" => LDA(method=:whiten, outdim=5) 
)

# Modelos con capas ocultas correctamente definidas
dic_modelos = Dict(
    "NeuralNetwork_50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(50,))),
    "NeuralNetwork_100" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100,))),
    "NeuralNetwork_100_50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100, 50))),
    
    "KNN_1" => KNNClassifier(K=1),
    "KNN_10" => KNNClassifier(K=10),
    "KNN_20" => KNNClassifier(K=20),

    "SVM_0.1" => SVC(cost=0.1),
    "SVM_0.5" => SVC(cost=0.5),
    "SVM_1" => SVC(cost=1.0)
);

In [3]:
function ejecutar_experimento(dic_filtros, dic_reducciones, dic_modelos, output_file; 
                              X=X_trainval, y=y_trainval, folds=folds_trainval)
    
    # Inicializamos DataFrame vacío (Siempre empieza de cero)
    results_df = DataFrame(
        Filter = String[], Reduction = String[], Model = String[],
        Accuracy = Float64[], F1_Score = Float64[]
    )
    
    # Métricas a evaluar
    measures = [accuracy, multiclass_f1score]

    # Bucle de ejecución
    for (filt_name, filt_model) in dic_filtros
        for (red_name, red_model) in dic_reducciones
            for (mod_name, mod_model) in dic_modelos
                
                println(">>> Evaluando: $filt_name + $red_name + $mod_name")
                
                # Construcción del Pipeline Manual
                scaler = MyMinMaxScaler()
                pipe = ManualPipeline(scaler, filt_model, red_model, mod_model)
                
                try
                    # Evaluación con Cross-Validation
                    evaluation = evaluate!(
                        pipe, X, y,
                        resampling = folds, measures = measures, verbosity = 0
                    )
                    
                    # Extraer métricas
                    acc = evaluation.measurement[1]
                    f1  = evaluation.measurement[2]
                    
                    # Guardar en DataFrame en memoria
                    push!(results_df, (filt_name, red_name, mod_name, acc, f1))
                    println("    Resultado: Acc=$acc | F1=$f1")
                    
                    # Guardado continuo
                    CSV.write(output_file, results_df)
                    
                catch e
                    println("!!! Error en $filt_name + $red_name + $mod_name: $e")
                    
                    # Registrar fallo como NaN
                    push!(results_df, (filt_name, red_name, mod_name, NaN, NaN))
                    
                    # Guardar también el error para tener constancia
                    CSV.write(output_file, results_df)
                end
            end
        end
    end
    
    println("Experimento finalizado.")
    return results_df
end

ejecutar_experimento (generic function with 1 method)

In [4]:
# Ejecutar y guardar
df_resultados_basicos = ejecutar_experimento(
    dic_filtros, 
    dic_reducciones, 
    dic_modelos, 
    "resultados_basicos.csv"
)

>>> Evaluando: MI + ICA + NeuralNetwork_100


Excessive output truncated after 10485804 bytes.

Row,Filter,Reduction,Model,Accuracy,F1_Score
,String,String,String,Float64,Float64
1,MI,ICA,NeuralNetwork_100,NaN,NaN
2,MI,ICA,KNN_10,NaN,NaN
3,MI,ICA,KNN_20,NaN,NaN
4,MI,ICA,NeuralNetwork_50,NaN,NaN
5,MI,ICA,NeuralNetwork_100_50,NaN,NaN
6,MI,ICA,SVM_0.1,NaN,NaN
7,MI,ICA,KNN_1,NaN,NaN
8,MI,ICA,SVM_0.5,NaN,NaN
9,MI,ICA,SVM_1,NaN,NaN
